In [ ]:
!uv pip install imlightgbm lightgbm numpy scikit-learn

## Binary classification

In [ ]:
import lightgbm as lgb
import numpy as np
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

import imlightgbm as imlgb

# Load breast cancer dataset
data = load_breast_cancer()
X, y = data.data, data.target

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create LightGBM datasets
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# Parameters for standard LightGBM model
params_standard = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42,
    "early_stopping_rounds": 10,
}

# Train standard LightGBM model
lgb_standard = lgb.train(
    params_standard, train_data, num_boost_round=100, valid_sets=[test_data]
)

# Parameters for Imbalanced LightGBM model
params_imbalanced = {
    "objective": "binary_focal",  # binary_weighted
    "gamma": 2.0,  # alpha with binary_weighted
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42,
    "early_stopping_rounds": 10,
}

# Train imbalanced LightGBM model
imlgb_focal = imlgb.train(
    params_imbalanced, train_data, num_boost_round=100, valid_sets=[test_data]
)

# Initialize the ImbalancedLGBMClassifier using binary focal loss
clf = imlgb.ImbalancedLGBMClassifier(
    objective="binary_weighted",  # binary_focal
    alpha=0.25,  # gamma with binary_focal
    learning_rate=0.05,
    num_leaves=31,
)

# Train the classifier on the training data
clf.fit(X=X_train, y=y_train)

# Predict using standard LightGBM model
y_pred_standard = lgb_standard.predict(X_test)
y_pred_standard_binary = (y_pred_standard > 0.5).astype(int)

# Predict using Imbalanced LightGBM model
y_pred_focal = imlgb_focal.predict(X_test)
y_pred_focal_binary = (y_pred_focal > 0.5).astype(int)

# Predict using ImbalancedLGBMClassifier
y_pred_weighted = clf.predict_proba(X_test)[:, 1]
y_pred_weighted_binary = clf.predict(X_test)

# Evaluate models
accuracy_standard = accuracy_score(y_test, y_pred_standard_binary)
logloss_standard = log_loss(y_test, y_pred_standard)
rocauc_standard = roc_auc_score(y_test, y_pred_standard)

accuracy_focal = accuracy_score(y_test, y_pred_focal_binary)
logloss_focal = log_loss(y_test, y_pred_focal)
rocauc_focal = roc_auc_score(y_test, y_pred_focal)

accuracy_weighted = accuracy_score(y_test, y_pred_weighted_binary)
logloss_weighted = log_loss(y_test, y_pred_weighted)
rocauc_weighted = roc_auc_score(y_test, y_pred_weighted)

# Print the evaluation results
print(
    f"Standard LightGBM - Accuracy: {accuracy_standard:.4f}, Log Loss: {logloss_standard:.4f}, rocauc: {rocauc_standard:.4f}"
)
print(
    f"LightGBM with Focal Loss - Accuracy: {accuracy_focal:.4f}, Log Loss: {logloss_focal:.4f}, rocauc: {rocauc_focal:.4f}"
)
print(
    f"LightGBM with Weighted Loss - Accuracy: {accuracy_weighted:.4f}, Log Loss: {logloss_weighted:.4f}, rocauc: {rocauc_weighted:.4f}"
)

## Multiclass classification

In [ ]:
# Generate dataset
X, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_classes=3,
    n_informative=5,
    weights=[0.05, 0.15, 0.8],
    flip_y=0,
    random_state=42,
)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create LightGBM datasets
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test)

# Parameters for standard LightGBM model
params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42,
    "early_stopping_rounds": 10,
}

# Train standard LightGBM model
lgb_standard = lgb.train(
    params, train_data, num_boost_round=100, valid_sets=[test_data]
)

# Parameters for Imbalanced LightGBM model
params = {
    "objective": "multiclass_focal",  # multiclass_weighted
    "num_class": 3,
    "gamma": 2.0,  # alpha with binary_weighted
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42,
    "early_stopping_rounds": 10,
}

# Train Imbalanced LightGBM model
imlgb_focal = imlgb.train(
    params, train_data, num_boost_round=100, valid_sets=[test_data]
)

# Initialize the ImbalancedLGBMClassifier
clf = imlgb.ImbalancedLGBMClassifier(
    objective="multiclass_weighted",  # multiclass_focal
    alpha=0.25,  # gamma with multiclass_focal
    num_class=3,
    learning_rate=0.05,
    num_leaves=31,
)

# Train the classifier on the training data
clf.fit(X=X_train, y=y_train)

# Predict using standard LightGBM model
y_pred_standard = lgb_standard.predict(X_test)
y_pred_standard_label = np.argmax(y_pred_standard, axis=1)

# Predict using Imbalanced LightGBM model
y_pred_focal = imlgb_focal.predict(X_test)
y_pred_focal_label = np.argmax(y_pred_focal, axis=1)

# Pridict using
y_pred_weighted = clf.predict_proba(X_test)
y_pred_weighted_label = clf.predict(X_test)

# Evaluate models
print("\nClassification Report for Standard:")
print(classification_report(y_test, y_pred_standard_label))

print("\nClassification Report for Focal Loss:")
print(classification_report(y_test, y_pred_focal_label))

print("\nClassification Report for Weighted loss:")
print(classification_report(y_test, y_pred_weighted_label))